## **Importing the Libraries**

In [27]:
from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Tokenizer, Qwen3Model, QWEN_CONFIG_06_B, KVCache
from pathlib import Path
from tqdm import tqdm
import time
import torch

## **Downloading and Loading the Tokenizer**

In [3]:

# downloading the tokenizer
file_path = Path("qwen3")/"tokenizer-base.json"
if not file_path.exists():
    download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")
else:
    print("Tokenizer is Already downloaded at path: ",file_path)

# loading the tokenizer
tokenizer_path = Path("qwen3")/"tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path = tokenizer_path)


Tokenizer is Already downloaded at path:  qwen3\tokenizer-base.json


## **Setting up the Device**

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.xpu.is_available():
        device = torch.device("xpu")
    else:
        device = torch.device("cpu")

    return device        


device(type='cpu')

In [ ]:
device = get_device()
device

## **Downloading the Model**

In [19]:
start_time = time.time()

file_path = (Path("qwen3")/"qwen3-0.6B-base.pth")

if not file_path.exists():
    download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")
else:
    print(f'Model is Already downloaded at path: {file_path}')
end_time = time.time()

print(f'Total time taken to download QWEN-3(small 0.6B): {end_time - start_time}')

Model is Already downloaded at path: qwen3\qwen3-0.6B-base.pth
Total time taken to download QWEN-3(small 0.6B): 0.0


In [20]:
model_path = Path('qwen3')/'qwen3-0.6B-base.pth'

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

## **Coding a minimal text generation function**

In [25]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None):

    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)
        
        if (eos_token_id is not None) and (torch.all(next_token == eos_token_id)):
            break

        yield next_token

        token_ids = torch.cat([token_ids, next_token], dim=1)



prompt = "Once upon a time, there was a village"
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device,).unsqueeze(0)

max_new_tokens=5

for token in generate_text_basic_stream(model, input_token_ids_tensor, max_new_tokens):
    token_id = token.squeeze(0).tolist()

    print(tokenizer.decode(token_id), end="", flush=True)



 in the middle of the

In [26]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    # tokenizer is currently unused but retained for backward compatibility
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 in the middle of the

Time: 8.32 sec
0 tokens/sec


## **Basic KV-Cache Text Generation**

In [30]:
@torch.inference_mode()
def generate_text_basic_stream_cache(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    cache = KVCache(n_layers=model.cfg['n_layers'])
    model.reset_kv_cache()
    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]


start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)



 in the middle of the

Time: 3.65 sec
1 tokens/sec
